# Hybrid Deep & Handcrafted Feature Fusion for Chest Radiograph Pneumonia Classification

**A Comparative Benchmark of Modern Vision Backbones (ConvNeXt-Tiny & Swin-T) with Domain-Specific Handcrafted Feature Descriptors**

---

### Abstract
Medical radiographs present subtle, multi-scale radiographic patterns (consolidation, ground-glass opacity, reticular interstitial infiltrates) that are challenging for standard deep networks due to texture bias and lack of explicit multi-frequency descriptors. 

This notebook implements an end-to-end clinical research benchmark evaluating whether fusing classic computer vision descriptors (**Multi-Scale HOG**, **Uniform LBP**, **24-Kernel Gabor Filter Banks**, and **SIFT-BoVW with TF-IDF**) into modern deep neural backbones (**ConvNeXt-Tiny** and **Swin Transformer-T**) yields superior classification performance, calibration, and interpretability on pediatric chest radiographs (Kermany cohort).

### Experimental Matrix:
- **Baseline 1:** ConvNeXt-Tiny (Pure Deep Representation, 768-D)
- **Model 2:** ConvNeXt-Tiny + Multi-Scale HOG (Projected L2-Normalized Concatenation, 460-D)
- **Model 3:** ConvNeXt-Tiny + Multi-Scale Uniform LBP (Gated Cross-Modal Bilinear Interaction, 270-D)
- **Baseline 2:** Swin Transformer-T (Pure Deep Representation, 768-D)
- **Model 5:** Swin Transformer-T + 24-Kernel Gabor Filter Bank (Projected L2-Normalized Concatenation, 288-D)
- **Model 6:** Swin Transformer-T + SIFT-BoVW TF-IDF (4-Head Token Cross-Attention + Squeeze-and-Excitation, 208-D)

---


## 1. Setup, Package Installation & Hardware Verification

In [ ]:
# Install required packages if missing in current environment
!pip install -q scikit-image huggingface_hub

import os
import io
import sys
import time
import json
import copy
import random
import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
from torchvision import transforms

from skimage.feature import hog, local_binary_pattern
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.manifold import TSNE
from huggingface_hub import hf_hub_download

import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Accelerator: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"PyTorch Version: {torch.__version__}")


## 2. Configuration & Seeding

In [ ]:
CONFIG = {
    "seed": 42,
    "img_size_deep": 224,
    "img_size_handcrafted": 128,
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid": (8, 8),
    "batch_size": 32,
    "epochs": 10,
    "optimizer": "AdamW",
    "backbone_lr": 1e-4,
    "head_lr": 5e-4,
    "weight_decay": 1e-2,
    "scheduler": "CosineAnnealingLR",
    "label_smoothing": 0.05,
    "embed_dim": 256,
    "dropout": 0.30,
    "grad_clip_norm": 1.0,
}

BASE_DIR = os.path.abspath(".")
DATA_DIR = os.path.join(BASE_DIR, "data", "chest_xray")
IMG_DIR = os.path.join(DATA_DIR, "images")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [DATA_DIR, IMG_DIR, OUT_DIR, CKPT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])
print("Configured output directories and initialized global seed:", CONFIG["seed"])


## 3. Dataset Download & Stratified Cohort Preparation
Downloads 1,500 clinical radiographs (600 Normal, 900 Pneumonia) directly from the Kermany Chest X-Ray pneumonia corpus via Hugging Face Hub, partitioning into a stratified 70% Train / 15% Validation / 15% Test split.

In [ ]:
MANIFEST_PATH = os.path.join(DATA_DIR, "manifest.csv")

def decode_pil_image(img_cell) -> np.ndarray:
    if isinstance(img_cell, dict) and "bytes" in img_cell:
        pil = Image.open(io.BytesIO(img_cell["bytes"])).convert("L")
    elif isinstance(img_cell, Image.Image):
        pil = img_cell.convert("L")
    else:
        raise TypeError(f"Unexpected image type: {type(img_cell)}")
    return np.array(pil, dtype=np.uint8)

def prepare_dataset():
    if os.path.exists(MANIFEST_PATH):
        print(f"Found existing manifest at {MANIFEST_PATH}. Loading cached dataset.")
        return pd.read_csv(MANIFEST_PATH)
    
    print("Fetching dataset shards from Hugging Face Hub (hf-vision/chest-xray-pneumonia)...")
    shards = [
        "data/train-00000-of-00007.parquet",
        "data/train-00002-of-00007.parquet",
        "data/validation-00000-of-00001.parquet",
        "data/test-00000-of-00001.parquet"
    ]
    dfs = []
    for s in shards:
        lp = hf_hub_download(repo_id="hf-vision/chest-xray-pneumonia", filename=s, repo_type="dataset")
        df_p = pd.read_parquet(lp)
        df_p["source_shard"] = os.path.basename(s)
        dfs.append(df_p)
    
    df_all = pd.concat(dfs, ignore_index=True)
    df_norm = df_all[df_all["label"] == 0].sample(n=min(600, (df_all["label"] == 0).sum()), random_state=CONFIG["seed"])
    df_pneu = df_all[df_all["label"] == 1].sample(n=min(900, (df_all["label"] == 1).sum()), random_state=CONFIG["seed"])
    df_sub = pd.concat([df_norm, df_pneu], ignore_index=True).sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
    
    y = df_sub["label"].values
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=CONFIG["seed"])
    train_idx, temp_idx = next(sss1.split(np.zeros(len(y)), y))
    
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=CONFIG["seed"])
    val_rel, test_rel = next(sss2.split(np.zeros(len(temp_idx)), y[temp_idx]))
    val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]
    
    splits = np.array([""] * len(df_sub), dtype=object)
    splits[train_idx] = "train"
    splits[val_idx] = "val"
    splits[test_idx] = "test"
    
    records = []
    print("Writing 256x256 standardized radiographs...")
    for idx, row in df_sub.iterrows():
        img_gray = decode_pil_image(row["image"])
        img_256 = cv2.resize(img_gray, (256, 256), interpolation=cv2.INTER_AREA)
        lbl = int(row["label"])
        cname = "PNEUMONIA" if lbl == 1 else "NORMAL"
        fn = f"{splits[idx]}_{cname.lower()}_{idx:04d}.png"
        cv2.imwrite(os.path.join(IMG_DIR, fn), img_256)
        records.append({
            "id": idx, "filename": fn, "rel_path": os.path.join("images", fn),
            "label": lbl, "class_name": cname, "split": splits[idx],
            "source_shard": row["source_shard"]
        })
    manifest = pd.DataFrame(records)
    manifest.to_csv(MANIFEST_PATH, index=False)
    print(f"Dataset successfully prepared: {len(manifest)} total radiographs.")
    return manifest

df_manifest = prepare_dataset()
for sp in ["train", "val", "test"]:
    n_sp = (df_manifest["split"] == sp).sum()
    n_nor = ((df_manifest["split"] == sp) & (df_manifest["label"] == 0)).sum()
    n_pne = ((df_manifest["split"] == sp) & (df_manifest["label"] == 1)).sum()
    print(f"  {sp.upper():5s} -> Total: {n_sp:4d} | Normal: {n_nor:4d} | Pneumonia: {n_pne:4d}")


## 4. Preprocessing & Handcrafted Feature Engineering
Implements:
1. **Symmetric CLAHE:** Dynamic histogram equalization preserving pulmonary contrast.
2. **Multi-Scale HOG (460-D):** Coarse (32x32) & Fine (16x16) gradient orientations + Sobel gradient stats.
3. **Multi-Scale Uniform LBP (270-D):** Uniform circular texture histograms across global and 2x2 spatial quadrants.
4. **24-Kernel Gabor Filter Bank (288-D):** 4 scales x 6 orientations frequency decomposition.
5. **SIFT-BoVW with TF-IDF (208-D):** Scale-Invariant Feature Transform with 128-cluster visual codebook and spatial pyramid statistics.

In [ ]:
class SymmetricCLAHE:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    def __call__(self, img_gray: np.ndarray) -> np.ndarray:
        if img_gray.ndim == 3:
            img_gray = cv2.cvtColor(img_gray, cv2.COLOR_RGB2GRAY)
        if img_gray.dtype != np.uint8:
            img_gray = np.clip(img_gray, 0, 255).astype(np.uint8)
        return self.clahe.apply(img_gray)

class HOGExtractor:
    def __init__(self, img_size=128):
        self.img_size = img_size
        self.scaler = StandardScaler()
    def extract_single(self, img_gray):
        img = cv2.resize(img_gray, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        hog_coarse = hog(img, orientations=9, pixels_per_cell=(32, 32), cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)
        hog_fine_map = hog(img, orientations=8, pixels_per_cell=(16, 16), cells_per_block=(1, 1), block_norm="L2-Hys", feature_vector=False).reshape(8, 8, 8)
        hog_fine = hog_fine_map.reshape(4, 2, 4, 2, 8).mean(axis=(1, 3)).flatten()
        gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx**2 + gy**2)
        ang = np.arctan2(gy, gx)
        stats = np.array([mag.mean(), mag.std(), np.percentile(mag, 75), np.percentile(mag, 90),
                          np.mean(np.abs(gx)), np.mean(np.abs(gy)), np.mean(np.cos(2*ang)), np.mean(np.sin(2*ang))], dtype=np.float32)
        return np.concatenate([hog_coarse, hog_fine, stats]).astype(np.float32)
    def fit_transform(self, images):
        raw = np.stack([self.extract_single(im) for im in images], axis=0)
        return self.scaler.fit_transform(raw).astype(np.float32)
    def transform(self, images):
        raw = np.stack([self.extract_single(im) for im in images], axis=0)
        return self.scaler.transform(raw).astype(np.float32)

class MultiScaleLBPExtractor:
    def __init__(self, img_size=128, scales=((8, 1), (16, 2), (24, 3))):
        self.img_size = img_size
        self.scales = scales
        self.scaler = StandardScaler()
    def _pyramid_hist(self, lbp_map, n_bins):
        h, w = lbp_map.shape
        mh, mw = h // 2, w // 2
        regs = [lbp_map, lbp_map[:mh, :mw], lbp_map[:mh, mw:], lbp_map[mh:, :mw], lbp_map[mh:, mw:]]
        hists = []
        for reg in regs:
            h_vals, _ = np.histogram(reg.ravel(), bins=n_bins, range=(0, n_bins), density=False)
            h_vals = h_vals.astype(np.float32)
            hists.append(h_vals / (h_vals.sum() + 1e-6))
        return np.concatenate(hists)
    def extract_single(self, img_gray):
        img = cv2.resize(img_gray, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        feats = [self._pyramid_hist(local_binary_pattern(img, P=P, R=R, method="uniform"), P + 2) for P, R in self.scales]
        return np.concatenate(feats).astype(np.float32)
    def fit_transform(self, images):
        raw = np.stack([self.extract_single(im) for im in images], axis=0)
        return self.scaler.fit_transform(raw).astype(np.float32)
    def transform(self, images):
        raw = np.stack([self.extract_single(im) for im in images], axis=0)
        return self.scaler.transform(raw).astype(np.float32)

class GaborFilterBankExtractor:
    def __init__(self, img_size=128):
        self.img_size = img_size
        self.wavelengths = [4.0, 8.0, 12.0, 16.0]
        self.orientations = [i * np.pi / 6.0 for i in range(6)]
        self.kernels = []
        for lambd in self.wavelengths:
            sigma = 0.56 * lambd
            ksize = int(max(7, round(sigma * 4) | 1))
            for theta in self.orientations:
                k = cv2.getGaborKernel((ksize, ksize), sigma=sigma, theta=theta, lambd=lambd, gamma=0.5, psi=0.0, ktype=cv2.CV_32F)
                k -= k.mean()
                k /= (np.linalg.norm(k) + 1e-6)
                self.kernels.append(k)
        self.scaler = StandardScaler()
    def extract_single(self, img_gray):
        img = cv2.resize(img_gray, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
        mh, mw = img.shape[0] // 2, img.shape[1] // 2
        feats = []
        for k in self.kernels:
            resp = np.abs(cv2.filter2D(img, cv2.CV_32F, k))
            feats.extend([resp.mean(), resp.std(), np.sqrt(np.mean(resp**2)), np.percentile(resp, 90)])
            for q in (resp[:mh, :mw], resp[:mh, mw:], resp[mh:, :mw], resp[mh:, mw:]):
                feats.extend([q.mean(), q.std()])
        return np.array(feats, dtype=np.float32)
    def fit_transform(self, images):
        raw = np.stack([self.extract_single(im) for im in images], axis=0)
        return self.scaler.fit_transform(raw).astype(np.float32)
    def transform(self, images):
        raw = np.stack([self.extract_single(im) for im in images], axis=0)
        return self.scaler.transform(raw).astype(np.float32)

class SIFTBoVWExtractor:
    def __init__(self, img_size=128, n_clusters=128, seed=42):
        self.img_size = img_size
        self.n_clusters = n_clusters
        self.seed = seed
        self.sift = cv2.SIFT_create(nfeatures=200)
        self.kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=2048, random_state=seed, n_init=3)
        self.scaler = StandardScaler()
        self.dense_kps = [cv2.KeyPoint(float(x), float(y), s) for y in range(16, img_size-8, 16) for x in range(16, img_size-8, 16) for s in (16.0, 24.0)]
    def _extract_sift_raw(self, img_gray):
        img = cv2.resize(img_gray, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        kps, desc = self.sift.detectAndCompute(img, None)
        _, dense_desc = self.sift.compute(img, self.dense_kps)
        half = self.img_size / 2.0
        q_counts = np.zeros(4, dtype=np.float32)
        q_resp = [[] for _ in range(4)]
        q_size = [[] for _ in range(4)]
        for kp in (kps or []):
            qi = (0 if kp.pt[1] < half else 2) + (0 if kp.pt[0] < half else 1)
            q_counts[qi] += 1.0; q_resp[qi].append(kp.response); q_size[qi].append(kp.size)
        stats = []
        for qi in range(4):
            stats.extend([q_counts[qi] / 50.0, float(np.mean(q_resp[qi])) if q_resp[qi] else 0.0,
                          float(np.mean(q_size[qi])) if q_size[qi] else 0.0, float(np.std(q_size[qi])) if len(q_size[qi]) > 1 else 0.0])
        descs, quads = [], []
        if desc is not None and len(desc) > 0:
            descs.append(desc); quads.extend([(0 if kp.pt[1] < half else 2) + (0 if kp.pt[0] < half else 1) for kp in kps])
        if dense_desc is not None and len(dense_desc) > 0:
            descs.append(dense_desc); quads.extend([(0 if kp.pt[1] < half else 2) + (0 if kp.pt[0] < half else 1) for kp in self.dense_kps])
        all_desc = np.vstack(descs).astype(np.float32)
        all_desc /= (all_desc.sum(axis=1, keepdims=True) + 1e-7)
        return np.sqrt(np.maximum(all_desc, 0.0)), np.array(quads, dtype=np.int32), np.array(stats, dtype=np.float32)
    def fit_transform(self, images):
        raw_list = [self._extract_sift_raw(im) for im in images]
        rng = np.random.RandomState(self.seed)
        sampled = [d[rng.choice(len(d), size=80, replace=False)] if len(d) > 80 else d for d, _, _ in raw_list]
        self.kmeans.fit(np.vstack(sampled))
        df = np.zeros(self.n_clusters, dtype=np.float32)
        for d, _, _ in raw_list:
            df[np.unique(self.kmeans.predict(d))] += 1.0
        self.idf = np.log((len(raw_list) + 1.0) / (df + 1.0)) + 1.0
        feats = self._encode_from_raw(raw_list)
        return self.scaler.fit_transform(feats).astype(np.float32)
    def _encode_from_raw(self, raw_list):
        encoded = []
        group_size = self.n_clusters // 16
        for desc, quads, stats in raw_list:
            words = self.kmeans.predict(desc)
            tf = np.bincount(words, minlength=self.n_clusters).astype(np.float32)
            tf /= (tf.sum() + 1e-6)
            tfidf = tf * self.idf
            tfidf /= (np.linalg.norm(tfidf) + 1e-6)
            sp_hists = []
            for qi in range(4):
                wq = words[quads == qi]
                if len(wq) > 0:
                    hq = np.bincount(wq // group_size, minlength=16).astype(np.float32)
                    hq /= (hq.sum() + 1e-6)
                else:
                    hq = np.zeros(16, dtype=np.float32)
                sp_hists.append(hq)
            encoded.append(np.concatenate([tfidf, np.concatenate(sp_hists), stats]).astype(np.float32))
        return np.stack(encoded, axis=0)
    def transform(self, images):
        raw_list = [self._extract_sift_raw(im) for im in images]
        return self.scaler.transform(self._encode_from_raw(raw_list)).astype(np.float32)


## 5. Precomputing Handcrafted Features with Synchronized Augmentations

In [ ]:
print("Loading radiographs and applying Symmetric CLAHE...")
clahe = SymmetricCLAHE(CONFIG["clahe_clip_limit"], CONFIG["clahe_tile_grid"])

splits_data = {}
for sp in ["train", "val", "test"]:
    sub = df_manifest[df_manifest["split"] == sp].reset_index(drop=True)
    imgs_orig, imgs_flip = [], []
    for rel_p in sub["rel_path"]:
        im = cv2.imread(os.path.join(DATA_DIR, rel_p), cv2.IMREAD_GRAYSCALE)
        eq = clahe(im)
        imgs_orig.append(eq)
        imgs_flip.append(cv2.flip(eq, 1))
    splits_data[sp] = {
        "df": sub,
        "imgs": imgs_orig,
        "imgs_flip": imgs_flip,
        "labels": sub["label"].values.astype(np.int64)
    }

print("Fitting handcrafted extractors on training radiographs...")
extractors = {
    "hog": HOGExtractor(img_size=CONFIG["img_size_handcrafted"]),
    "lbp": MultiScaleLBPExtractor(img_size=CONFIG["img_size_handcrafted"]),
    "gabor": GaborFilterBankExtractor(img_size=CONFIG["img_size_handcrafted"]),
    "sift_bovw": SIFTBoVWExtractor(img_size=CONFIG["img_size_handcrafted"], n_clusters=128, seed=CONFIG["seed"]),
}

hand_features = {}
for name, ext in extractors.items():
    t0 = time.time()
    tr_orig = ext.fit_transform(splits_data["train"]["imgs"])
    tr_flip = ext.transform(splits_data["train"]["imgs_flip"])
    va_orig = ext.transform(splits_data["val"]["imgs"])
    te_orig = ext.transform(splits_data["test"]["imgs"])
    dt = time.time() - t0
    hand_features[name] = {
        "train_orig": tr_orig, "train_flip": tr_flip,
        "val": va_orig, "test": te_orig,
        "dim": int(tr_orig.shape[1]), "time": dt
    }
    print(f"  {name:10s} -> Feature Dim: {tr_orig.shape[1]:3d} | Precomputed in {dt:.1f}s")


## 6. Hybrid Medical Classifier & Cross-Modal Fusion Heads

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class HybridChestXRayDataset(Dataset):
    def __init__(self, clahe_images, labels, hand_orig=None, hand_flip=None, is_train=False, img_size=224):
        self.clahe_images = clahe_images
        self.labels = np.asarray(labels, dtype=np.int64)
        self.hand_orig = hand_orig
        self.hand_flip = hand_flip
        self.is_train = is_train
        if is_train:
            self.spatial_tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomRotation(degrees=7, fill=0),
                transforms.ColorJitter(brightness=0.10, contrast=0.10),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ])
        else:
            self.spatial_tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ])

    def __len__(self):
        return len(self.clahe_images)

    def __getitem__(self, idx):
        im = self.clahe_images[idx]
        flipped = False
        if self.is_train and random.random() < 0.5:
            im = cv2.flip(im, 1)
            flipped = True
        img_t = self.spatial_tf(Image.fromarray(cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)))
        if self.hand_orig is not None:
            vec = self.hand_flip[idx] if (flipped and self.hand_flip is not None) else self.hand_orig[idx]
            hand_t = torch.from_numpy(vec).float()
        else:
            hand_t = torch.zeros(1, dtype=torch.float32)
        return img_t, hand_t, int(self.labels[idx])

# Fusion Module Implementations
class PureDeepHead(nn.Module):
    def __init__(self, deep_dim, embed_dim=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(deep_dim), nn.Dropout(dropout),
            nn.Linear(deep_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(embed_dim, num_classes))
    def forward(self, deep_feat, hand_feat=None):
        emb = self.proj(deep_feat)
        return self.classifier(emb), emb, None

class ProjectedConcatFusion(nn.Module):
    def __init__(self, deep_dim, hand_dim, embed_dim=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.deep_proj = nn.Sequential(
            nn.LayerNorm(deep_dim), nn.Dropout(dropout),
            nn.Linear(deep_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.hand_proj = nn.Sequential(
            nn.LayerNorm(hand_dim), nn.Dropout(dropout),
            nn.Linear(hand_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.fusion_mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.LayerNorm(embed_dim), nn.GELU(), nn.Dropout(0.2)
        )
        self.classifier = nn.Linear(embed_dim, num_classes)
    def forward(self, deep_feat, hand_feat):
        scale = (256 ** 0.5) * 0.25
        f_d = F.normalize(self.deep_proj(deep_feat), p=2, dim=-1) * scale
        f_h = F.normalize(self.hand_proj(hand_feat), p=2, dim=-1) * scale
        emb = self.fusion_mlp(torch.cat([f_d, f_h], dim=-1)) + f_d
        return self.classifier(emb), emb, None

class GatedCrossModalFusion(nn.Module):
    def __init__(self, deep_dim, hand_dim, embed_dim=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.deep_proj = nn.Sequential(
            nn.LayerNorm(deep_dim), nn.Dropout(dropout),
            nn.Linear(deep_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.hand_proj = nn.Sequential(
            nn.LayerNorm(hand_dim), nn.Dropout(dropout),
            nn.Linear(hand_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.gate_net = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
        self.interaction_proj = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.Tanh())
        self.out_norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(embed_dim, num_classes))
    def forward(self, deep_feat, hand_feat):
        f_d = self.deep_proj(deep_feat)
        f_h = self.hand_proj(hand_feat)
        alpha = self.gate_net(torch.cat([f_d, f_h], dim=-1))
        interaction = self.interaction_proj(f_d * f_h)
        emb = self.out_norm(f_d + alpha * f_h + (1.0 - alpha) * 0.25 * interaction)
        return self.classifier(emb), emb, alpha.mean(dim=-1)

class CrossModalTokenAttentionFusion(nn.Module):
    def __init__(self, deep_dim, hand_dim, embed_dim=256, num_heads=4, num_classes=2, dropout=0.3):
        super().__init__()
        self.deep_proj = nn.Sequential(
            nn.LayerNorm(deep_dim), nn.Dropout(dropout),
            nn.Linear(deep_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.hand_proj = nn.Sequential(
            nn.LayerNorm(hand_dim), nn.Dropout(dropout),
            nn.Linear(hand_dim, embed_dim), nn.LayerNorm(embed_dim), nn.GELU()
        )
        self.mod_pos = nn.Parameter(torch.randn(1, 2, embed_dim) * 0.02)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mhsa = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.fuse_proj = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.se = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 4), nn.GELU(),
            nn.Linear(embed_dim // 4, embed_dim), nn.Sigmoid()
        )
        self.out_norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(embed_dim, num_classes))
    def forward(self, deep_feat, hand_feat):
        f_d = self.deep_proj(deep_feat)
        f_h = self.hand_proj(hand_feat)
        tokens = torch.stack([f_d, f_h], dim=1) + self.mod_pos
        normed = self.norm1(tokens)
        attn_out, attn_weights = self.mhsa(normed, normed, normed, need_weights=True, average_attn_weights=True)
        tokens = tokens + attn_out
        joined = self.fuse_proj(tokens.reshape(tokens.size(0), -1))
        emb = self.out_norm(f_d + joined * self.se(joined))
        return self.classifier(emb), emb, attn_weights[:, 0, 1]

class HybridMedicalClassifier(nn.Module):
    def __init__(self, backbone_name="convnext_tiny", fusion_type="none", hand_dim=0, embed_dim=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.fusion_type = fusion_type.lower()
        if self.backbone_name == "convnext_tiny":
            base = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            self.frozen_stages = base.features[:6]
            self.trainable_stage = base.features[6:]
            self.post_norm = nn.Identity()
            self.avgpool = base.avgpool
            self.deep_dim = 768
        elif self.backbone_name == "swin_t":
            base = tv_models.swin_t(weights=tv_models.Swin_T_Weights.IMAGENET1K_V1)
            self.frozen_stages = base.features[:6]
            self.trainable_stage = base.features[6:]
            self.post_norm = base.norm
            self.avgpool = base.avgpool
            self.deep_dim = 768
        for p in self.frozen_stages.parameters():
            p.requires_grad = False
        self.frozen_stages.eval()

        if self.fusion_type == "none":
            self.head = PureDeepHead(self.deep_dim, embed_dim, num_classes, dropout)
        elif self.fusion_type == "projected_concat":
            self.head = ProjectedConcatFusion(self.deep_dim, hand_dim, embed_dim, num_classes, dropout)
        elif self.fusion_type == "gated_attention":
            self.head = GatedCrossModalFusion(self.deep_dim, hand_dim, embed_dim, num_classes, dropout)
        elif self.fusion_type == "token_attention":
            self.head = CrossModalTokenAttentionFusion(self.deep_dim, hand_dim, embed_dim, num_heads=4, num_classes=num_classes, dropout=dropout)

    def train(self, mode=True):
        super().train(mode)
        self.frozen_stages.eval()
        return self

    def extract_deep_features(self, x):
        with torch.no_grad():
            h = self.frozen_stages(x)
        h = self.trainable_stage(h)
        if self.backbone_name == "swin_t":
            h = self.post_norm(h).permute(0, 3, 1, 2)
        return torch.flatten(self.avgpool(h), 1)

    def forward(self, x, hand_feat=None):
        return self.head(self.extract_deep_features(x), hand_feat)

    def get_param_groups(self, backbone_lr=1e-4, head_lr=5e-4, weight_decay=1e-2):
        bb = [p for p in list(self.trainable_stage.parameters()) + list(self.post_norm.parameters()) if p.requires_grad]
        hd = [p for p in self.head.parameters() if p.requires_grad]
        return [{"params": bb, "lr": backbone_lr, "weight_decay": weight_decay},
                {"params": hd, "lr": head_lr, "weight_decay": weight_decay}]


## 7. Model Training & Clinical Evaluation Framework (Mixed Precision AMP)

In [ ]:
def compute_clinical_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred) * 100.0), 2),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0) * 100.0), 2),
        "recall_sensitivity": round(float(recall_score(y_true, y_pred, zero_division=0) * 100.0), 2),
        "specificity": round(float(spec * 100.0), 2),
        "f1_score": round(float(f1_score(y_true, y_pred, zero_division=0) * 100.0), 2),
        "f1_macro": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0) * 100.0), 2),
        "auroc": round(float(roc_auc_score(y_true, y_prob)), 4),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "roc_curve": {"fpr": [round(float(x), 4) for x in fpr], "tpr": [round(float(x), 4) for x in tpr]},
    }

@torch.no_grad()
def evaluate_model(model, loader, criterion, device, collect_artifacts=False):
    model.eval()
    total_loss, n_samples = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    all_embs, all_attn = [], []
    for imgs, h_feats, labels in loader:
        imgs, h_feats, labels = imgs.to(device), h_feats.to(device), labels.to(device)
        logits, emb, attn_info = model(imgs, h_feats)
        loss = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        n_samples += bs
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
        y_prob.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy().tolist())
        if collect_artifacts:
            all_embs.append(emb.cpu().numpy())
            if attn_info is not None:
                all_attn.append(attn_info.cpu().numpy())
    metrics = compute_clinical_metrics(y_true, y_pred, y_prob)
    metrics["loss"] = round(float(total_loss / max(1, n_samples)), 4)
    artifacts = None
    if collect_artifacts:
        artifacts = {
            "y_true": np.array(y_true, dtype=np.int64),
            "y_pred": np.array(y_pred, dtype=np.int64),
            "y_prob": np.array(y_prob, dtype=np.float32),
            "embeddings": np.concatenate(all_embs, axis=0) if all_embs else None,
            "attn_weights": np.concatenate(all_attn, axis=0) if all_attn else None,
        }
    return metrics, artifacts

def run_single_experiment(spec, splits, hand_features, device):
    exp_id = spec["id"]
    print(f"\n{'='*60}\nRunning [{exp_id}]: {spec['title']}\n{'='*60}")
    seed_everything(CONFIG["seed"])
    f_key = spec["feature"]
    if f_key == "none":
        h_dim, tr_o, tr_f, va, te = 0, None, None, None, None
    else:
        fd = hand_features[f_key]
        h_dim, tr_o, tr_f, va, te = fd["dim"], fd["train_orig"], fd["train_flip"], fd["val"], fd["test"]

    train_ds = HybridChestXRayDataset(splits["train"]["imgs"], splits["train"]["labels"], tr_o, tr_f, True, CONFIG["img_size_deep"])
    val_ds = HybridChestXRayDataset(splits["val"]["imgs"], splits["val"]["labels"], va, None, False, CONFIG["img_size_deep"])
    test_ds = HybridChestXRayDataset(splits["test"]["imgs"], splits["test"]["labels"], te, None, False, CONFIG["img_size_deep"])

    g = torch.Generator().manual_seed(CONFIG["seed"])
    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, generator=g)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

    model = HybridMedicalClassifier(spec["backbone"], spec["fusion"], h_dim, CONFIG["embed_dim"], 2, CONFIG["dropout"]).to(device)
    tr_labels = splits["train"]["labels"]
    n_norm, n_pneu = float((tr_labels == 0).sum()), float((tr_labels == 1).sum())
    weights = torch.tensor([(n_norm + n_pneu) / (2.0 * n_norm), (n_norm + n_pneu) / (2.0 * n_pneu)], dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=CONFIG["label_smoothing"])
    optimizer = torch.optim.AdamW(model.get_param_groups(CONFIG["backbone_lr"], CONFIG["head_lr"], CONFIG["weight_decay"]))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)
    use_amp = (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1_macro": [], "val_auroc": []}
    best_score, best_epoch, best_state = -1.0, 0, None
    t0 = time.time()

    for epoch in range(1, CONFIG["epochs"] + 1):
        ep_t0 = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, h_feats, labels in train_loader:
            imgs, h_feats, labels = imgs.to(device), h_feats.to(device), labels.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits, _, _ = model(imgs, h_feats)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
                scaler.step(optimizer)
                scaler.update()
            else:
                logits, _, _ = model(imgs, h_feats)
                loss = criterion(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
                optimizer.step()
            bs = labels.size(0)
            running_loss += loss.item() * bs
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += bs

        scheduler.step()
        train_loss = running_loss / max(1, total)
        train_acc = (correct / max(1, total)) * 100.0
        val_m, _ = evaluate_model(model, val_loader, criterion, device, collect_artifacts=False)

        history["train_loss"].append(round(train_loss, 4))
        history["train_acc"].append(round(train_acc, 2))
        history["val_loss"].append(val_m["loss"])
        history["val_acc"].append(val_m["accuracy"])
        history["val_f1_macro"].append(val_m["f1_macro"])
        history["val_auroc"].append(val_m["auroc"])

        score = 0.5 * val_m["f1_macro"] + 0.5 * (val_m["auroc"] * 100.0)
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Ep {epoch:02d}/{CONFIG['epochs']:02d} ({time.time()-ep_t0:4.1f}s) | "
              f"TrAcc: {train_acc:5.2f}% | ValAcc: {val_m['accuracy']:5.2f}% | ValAUC: {val_m['auroc']:.4f}")

    total_sec = round(time.time() - t0, 1)
    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(CKPT_DIR, f"{exp_id}.pth"))

    val_best_m, _ = evaluate_model(model, val_loader, criterion, device, collect_artifacts=False)
    test_m, test_art = evaluate_model(model, test_loader, criterion, device, collect_artifacts=True)

    attn_summary = None
    if test_art["attn_weights"] is not None:
        aw, yt = test_art["attn_weights"], test_art["y_true"]
        attn_summary = {
            "overall_mean": round(float(aw.mean()), 4),
            "normal_mean": round(float(aw[yt == 0].mean()), 4),
            "pneumonia_mean": round(float(aw[yt == 1].mean()), 4),
        }
    print(f"  --> Final Test [{exp_id}] | Acc: {test_m['accuracy']:.2f}% | F1: {test_m['f1_score']:.2f}% | AUROC: {test_m['auroc']:.4f}")

    return {
        "id": exp_id, "title": spec["title"], "backbone": spec["backbone"],
        "feature_extractor": spec["feature_label"], "feature_key": spec["feature"],
        "feature_dim": h_dim, "fusion_strategy": spec["fusion_label"],
        "best_epoch": int(best_epoch), "train_time_sec": total_sec,
        "val_metrics": val_best_m, "test_metrics": test_m,
        "history": history, "attn_summary": attn_summary,
    }, test_art


## 8. Master Benchmark Execution Across All 6 Configurations

In [ ]:
experiments = [
    {
        "id": "exp1_convnext_baseline",
        "title": "ConvNeXt-Tiny (Pure Deep Baseline)",
        "backbone": "convnext_tiny",
        "feature": "none",
        "feature_label": "None (Pure Deep Features)",
        "fusion": "none",
        "fusion_label": "None (Baseline Head)",
    },
    {
        "id": "exp2_convnext_hog",
        "title": "ConvNeXt-Tiny + Multi-Scale HOG (Projected Concat)",
        "backbone": "convnext_tiny",
        "feature": "hog",
        "feature_label": "Multi-Scale Spatial HOG (460-D)",
        "fusion": "projected_concat",
        "fusion_label": "Projected Dimensionality-Aligned Concatenation",
    },
    {
        "id": "exp3_convnext_lbp",
        "title": "ConvNeXt-Tiny + Multi-Scale Uniform LBP (Gated Cross-Modal)",
        "backbone": "convnext_tiny",
        "feature": "lbp",
        "feature_label": "Multi-Scale Pyramid Uniform LBP (270-D)",
        "fusion": "gated_attention",
        "fusion_label": "Gated Cross-Modal Attention Fusion",
    },
    {
        "id": "exp4_swin_baseline",
        "title": "Swin-T (Pure Deep Baseline)",
        "backbone": "swin_t",
        "feature": "none",
        "feature_label": "None (Pure Deep Features)",
        "fusion": "none",
        "fusion_label": "None (Baseline Head)",
    },
    {
        "id": "exp5_swin_gabor",
        "title": "Swin-T + 24-Kernel Gabor Filter Bank (Projected Concat)",
        "backbone": "swin_t",
        "feature": "gabor",
        "feature_label": "24-Kernel Gabor Filter Bank (288-D)",
        "fusion": "projected_concat",
        "fusion_label": "Projected Dimensionality-Aligned Concatenation",
    },
    {
        "id": "exp6_swin_sift",
        "title": "Swin-T + SIFT-BoVW TF-IDF (Token Attention + SE)",
        "backbone": "swin_t",
        "feature": "sift_bovw",
        "feature_label": "SIFT + K-Means BoVW TF-IDF (208-D)",
        "fusion": "token_attention",
        "fusion_label": "Multi-Head Cross-Modal Token Attention + SE",
    },
]

all_results, artifact_bundle = [], {}
for spec in experiments:
    res, art = run_single_experiment(spec, splits_data, hand_features, device)
    all_results.append(res)
    artifact_bundle[f"{spec['id']}_y_true"] = art["y_true"]
    artifact_bundle[f"{spec['id']}_y_pred"] = art["y_pred"]
    artifact_bundle[f"{spec['id']}_y_prob"] = art["y_prob"]
    if art["embeddings"] is not None:
        artifact_bundle[f"{spec['id']}_emb"] = art["embeddings"]
    if art["attn_weights"] is not None:
        artifact_bundle[f"{spec['id']}_attn"] = art["attn_weights"]

payload = {
    "config": CONFIG,
    "feature_dims": {k: v["dim"] for k, v in hand_features.items()},
    "experiments": all_results,
}
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(payload, f, indent=4)
np.savez_compressed(os.path.join(OUT_DIR, "eval_artifacts.npz"), **artifact_bundle)
print("\nAll 6 experimental runs successfully completed!")


## 9. Quantitative Benchmark Summary Tables

In [ ]:
summary_rows = []
for r in all_results:
    tm = r["test_metrics"]
    cm = tm["confusion_matrix"]
    summary_rows.append({
        "Experiment": r["id"],
        "Architecture & Fusion": r["title"],
        "Feature Dim": r["feature_dim"],
        "Accuracy (%)": tm["accuracy"],
        "Sensitivity (%)": tm["recall_sensitivity"],
        "Specificity (%)": tm["specificity"],
        "Precision (%)": tm["precision"],
        "F1-Score (%)": tm["f1_score"],
        "AUROC": tm["auroc"],
        "TN": cm["tn"], "FP": cm["fp"], "FN": cm["fn"], "TP": cm["tp"]
    })

df_summary = pd.DataFrame(summary_rows)
print("=== CLINICAL PERFORMANCE BENCHMARK MATRIX ===")
display(df_summary[["Architecture & Fusion", "Feature Dim", "Accuracy (%)", "Sensitivity (%)", "Specificity (%)", "F1-Score (%)", "AUROC"]])


## 10. High-Resolution Visualizations & Publication Figures
Generates and displays:
1. Symmetric CLAHE & Handcrafted Descriptors.
2. Test Confusion Matrices.
3. Comparative ROC Curves & AUROC scores.
4. Grouped Clinical Performance Metrics.
Packages all results and figures into a downloadable ZIP archive.

In [ ]:
# 1. CLAHE & Handcrafted pipeline
test_df = df_manifest[df_manifest["split"] == "test"]
norm_row = test_df[test_df["label"] == 0].iloc[2]
pneu_row = test_df[test_df["label"] == 1].iloc[2]

sift = cv2.SIFT_create(nfeatures=120)
gabor_kernels = [cv2.getGaborKernel((15, 15), 4.0, th, 8.0, 0.5, 0.0, ktype=cv2.CV_32F) for th in [0, np.pi/4, np.pi/2, 3*np.pi/4]]

fig, axes = plt.subplots(2, 6, figsize=(16, 5.6))
col_titles = ["1. Raw Radiograph", "2. Symmetric CLAHE", "3. HOG Gradients", "4. Uniform LBP (R=2)", "5. Gabor Energy", "6. SIFT Keypoints"]
for row_i, (row_label, row) in enumerate([("NORMAL\n(Healthy)", norm_row), ("PNEUMONIA\n(Infiltrate)", pneu_row)]):
    raw = cv2.imread(os.path.join(DATA_DIR, row["rel_path"]), cv2.IMREAD_GRAYSCALE)
    eq = clahe(raw)
    eq128 = cv2.resize(eq, (128, 128), interpolation=cv2.INTER_AREA)
    _, hog_vis = hog(eq128, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True, block_norm="L2-Hys")
    lbp_map = local_binary_pattern(eq128, P=16, R=2, method="uniform")
    f_img = eq128.astype(np.float32) / 255.0
    g_resp = np.mean([np.abs(cv2.filter2D(f_img, cv2.CV_32F, k)) for k in gabor_kernels], axis=0)
    kps, _ = sift.detectAndCompute(eq128, None)
    sift_vis = cv2.drawKeypoints(eq128, kps[:80], None, color=(255, 80, 40), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    panels = [(raw, "gray"), (eq, "gray"), (hog_vis, "inferno"), (lbp_map, "viridis"), (g_resp, "magma"), (sift_vis, None)]
    for col_i, (im_show, cmap) in enumerate(panels):
        ax = axes[row_i, col_i]
        ax.imshow(im_show, cmap=cmap) if cmap else ax.imshow(im_show)
        ax.set_xticks([]); ax.set_yticks([])
        if row_i == 0: ax.set_title(col_titles[col_i], fontsize=9.5)
        if col_i == 0: ax.set_ylabel(row_label, fontsize=10, fontweight="bold")

fig.suptitle("Symmetric CLAHE and Four Multi-Scale Handcrafted Feature Extractors", fontsize=12.5, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig1_preprocessing_handcrafted_pipeline.png"), dpi=300, bbox_inches="tight")
plt.show()

# 2. Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(14, 8.2))
for idx, r in enumerate(all_results):
    ax = axes[idx // 3, idx % 3]
    cm_d = r["test_metrics"]["confusion_matrix"]
    mat = np.array([[cm_d["tn"], cm_d["fp"]], [cm_d["fn"], cm_d["tp"]]])
    pct = mat / np.maximum(mat.sum(axis=1, keepdims=True), 1) * 100.0
    ax.imshow(mat, cmap="Blues" if idx < 3 else "Purples", vmin=0, vmax=mat.max() * 1.15)
    for i in range(2):
        for j in range(2):
            val = mat[i, j]
            ax.text(j, i, f"{val}\n({pct[i, j]:.1f}%)", ha="center", va="center",
                    fontsize=10.5, fontweight="bold", color="white" if val > mat.max() * 0.55 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred: Normal", "Pred: Pneumonia"]); ax.set_yticklabels(["True: Normal", "True: Pneumonia"])
    ax.set_title(f"{r['title']}\nAcc: {r['test_metrics']['accuracy']:.2f}% | F1: {r['test_metrics']['f1_score']:.2f}%", fontsize=9.2)

fig.suptitle("Held-Out Test Confusion Matrices (n = 225 Radiographs)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig4_confusion_matrices_all.png"), dpi=300, bbox_inches="tight")
plt.show()

# 3. ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
colors = ["#2E4053", "#1F618D", "#C0392B"]
lss = ["--", "-", "-"]
for ax, grp, title, lbls in [
    (axes[0], all_results[:3], "ConvNeXt-Tiny Series", ["Baseline", "+ HOG (Concat)", "+ LBP (Gated Attn)"]),
    (axes[1], all_results[3:], "Swin-T Series", ["Baseline", "+ Gabor (Concat)", "+ SIFT-BoVW (Token Attn)"])
]:
    for i, (r, lbl) in enumerate(zip(grp, lbls)):
        roc = r["test_metrics"]["roc_curve"]
        ax.plot(roc["fpr"], roc["tpr"], color=colors[i], linestyle=lss[i], lw=2.3, label=f"{lbl} (AUROC = {r['test_metrics']['auroc']:.4f})")
    ax.plot([0, 1], [0, 1], "k:", lw=1.2, alpha=0.6, label="Random (0.5000)")
    ax.set_title(title); ax.set_xlabel("FPR (1 - Specificity)"); ax.set_ylabel("TPR (Sensitivity)")
    ax.grid(True, linestyle="--", alpha=0.4); ax.legend(loc="lower right", fontsize=8.8)

fig.suptitle("Test Set ROC Curves Across Architecture Configurations", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig5_roc_curves_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

# 4. Grouped Metrics
labels = ["ConvNeXt\n(Base)", "ConvNeXt\n+HOG", "ConvNeXt\n+LBP", "Swin-T\n(Base)", "Swin-T\n+Gabor", "Swin-T\n+SIFT"]
metrics_keys = [("accuracy", "Accuracy", "#1B4F72"), ("precision", "Precision", "#2874A6"),
                ("recall_sensitivity", "Sensitivity", "#117A65"), ("specificity", "Specificity", "#B7950B"), ("f1_score", "F1", "#A93226")]
x = np.arange(len(all_results))
fig, ax = plt.subplots(figsize=(14, 5.4))
for m_i, (k, name, col) in enumerate(metrics_keys):
    vals = [r["test_metrics"][k] for r in all_results]
    bars = ax.bar(x + (m_i - 2) * 0.155, vals, 0.155, label=name, color=col, edgecolor="white")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.5, f"{v:.1f}", ha="center", va="bottom", fontsize=7.2, rotation=90)
ax.axvline(2.5, color="#7F8C8D", linestyle="--", lw=1.2)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.2); ax.set_ylabel("Score (%)"); ax.set_ylim(70, 109)
ax.grid(True, axis="y", linestyle="--", alpha=0.4); ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.22))
ax.set_title("Clinical Test Evaluation Metrics Across Architecture Configurations", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig6_grouped_metrics_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

# Package artifacts and figures into archive
import zipfile
zip_path = os.path.join(BASE_DIR, "outputs_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(os.path.join(OUT_DIR, "results.json"), "results.json")
    zf.write(os.path.join(OUT_DIR, "eval_artifacts.npz"), "eval_artifacts.npz")
    for root, _, files in os.walk(FIG_DIR):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.join("figures", f))

print(f"\nAll tasks complete! Results and figures archived in '{zip_path}'.")
